In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('cleaned_hospital_data_final.parquet')

In [2]:
sum(df.duplicated())

0

In [3]:
df.isna().sum().sum()

0

In [4]:
for col in df.select_dtypes(include=['object', 'category']).columns:
    blank_mask = df[col].str.strip().eq('').fillna(False)

    if blank_mask.any():
        print(f"Warning: {col} contains {blank_mask.sum()} blank values")

In [5]:
for col in ['Total Charges', 'Total Costs']:
    if df[col].dtype == 'object':
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('$', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Zero or Negative Charges:", (df['Total Charges'] <= 0).sum())
print("Zero or Negative Costs:", (df['Total Costs'] <= 0).sum())

Zero or Negative Charges: 1
Zero or Negative Costs: 1


In [6]:
for col in ['Total Charges', 'Total Costs']:
    print(f"\n{col}")
    print("Zero:", (df[col] == 0).sum())
    print("Negative:", (df[col] < 0).sum())
    print("Missing:", df[col].isna().sum())

problem_rows = df[
    (df['Total Charges'] <= 0) |
    (df['Total Costs'] <= 0) |
    (df['Total Charges'].isna()) |
    (df['Total Costs'].isna())
]

display(problem_rows.head(10))


Total Charges
Zero: 1
Negative: 0
Missing: 0

Total Costs
Zero: 1
Negative: 0
Missing: 0


,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs,Is_Neonate,Has_Secondary_Insurance
652605,Western NY,Niagara,3121001.0,583.0,Mount St. Mary's Hospital and Health Center,30 to 49,142,F,White,Not Span/Hispanic,...,Moderate,Minor,Medical,Medicaid,0.0,N,0.0,0.0,0,0


In [7]:
print("Data type of Length of Stay:", df['Length of Stay'].dtype)
if df['Length of Stay'].dtype == 'object':
    print("Unique non-numeric values:", df[~df['Length of Stay'].astype(str).str.isnumeric()]['Length of Stay'].unique())

Data type of Length of Stay: object
Unique non-numeric values: ['120 +']


In [8]:
df['Length of Stay'] = df['Length of Stay'].replace('120 +', '120')

df['Length of Stay'] = pd.to_numeric(df['Length of Stay'], errors='coerce').astype('int32')

print("New Data Type:", df['Length of Stay'].dtype)
print("Max Stay:", df['Length of Stay'].max())
print("Min Stay:", df['Length of Stay'].min())

New Data Type: int32
Max Stay: 120
Min Stay: 1


In [9]:
df['Discharge Year'].value_counts()

Discharge Year
2023    2115867
2022    2092832
Name: count, dtype: int64

In [10]:
cols_to_check = ['Gender', 'Race', 'Ethnicity', 'Age Group', 'Type of Admission']
for col in cols_to_check:
    print(f"{col}")
    print(df[col].unique())
    print()

Gender
['M' 'F' 'U']

Race
['Other Race' 'White' 'Black/African American' 'Multi-racial']

Ethnicity
['Spanish/Hispanic' 'Unknown' 'Not Span/Hispanic' 'Multi-ethnic']

Age Group
['18 to 29' '50 to 69' '70 or Older' '30 to 49' '0 to 17']

Type of Admission
['Emergency' 'Elective' 'Urgent' 'Not Available' 'Newborn' 'Trauma']



In [11]:
print("Emergency Indicator values:\n", df['Emergency Department Indicator'].value_counts())

Emergency Indicator values:
 Emergency Department Indicator
Y    2665626
N    1543073
Name: count, dtype: int64


In [12]:
abnormal_costs = (df['Total Costs'] > df['Total Charges']).sum()
print("Cases where Costs > Charges:", abnormal_costs)

Cases where Costs > Charges: 52507


In [13]:
extreme_cases = df[df['Total Costs'] > (df['Total Charges'] * 5)]
print("Extreme anomalies count:", len(extreme_cases))

Extreme anomalies count: 130


In [14]:
extreme_cases = df[
    df['Total Costs'] > df['Total Charges'] * 5
]

print("Extreme anomalies:", len(extreme_cases))

display(
    extreme_cases[
        ['Total Charges', 'Total Costs', 'Length of Stay',
         'Age Group', 'Type of Admission',
         'Patient Disposition', 'Discharge Year']
    ].sort_values('Total Costs', ascending=False)
)

Extreme anomalies: 130


,Total Charges,Total Costs,Length of Stay,Age Group,Type of Admission,Patient Disposition,Discharge Year
1506586,120261.44,3979133.10,8,70 or Older,Emergency,Facility w/ Custodial/Supportive Care,2022
351780,74936.75,3970456.20,3,50 to 69,Emergency,Home or Self Care,2022
3989748,414261.99,2424558.67,9,30 to 49,Elective,Home w/ Home Health Services,2023
2668318,298829.85,2393557.83,6,50 to 69,Elective,Home or Self Care,2023
2668230,297161.83,2389780.21,6,70 or Older,Elective,Home or Self Care,2023
...,...,...,...,...,...,...,...
415066,5914.00,49395.64,1,0 to 17,Emergency,Home or Self Care,2022
602912,5914.00,49395.64,1,0 to 17,Urgent,Home or Self Care,2022
164560,5761.00,49375.52,1,0 to 17,Elective,Home or Self Care,2022
3412410,4408.56,22378.16,4,70 or Older,Emergency,Hosp Basd Medicare Approved Swing Bed,2023


In [15]:
# How many times Total Costs exceed Total Charges
df['Cost to Charge Ratio'] = (
    df['Total Costs'] / df['Total Charges']
)

# Absolute difference between costs and charges
df['Cost Charge Difference'] = (
    df['Total Costs'] - df['Total Charges']
)

# Show the 10 cases with the highest ratio
display(
    df.nlargest(10, 'Cost to Charge Ratio')[
        [
            'Total Charges',
            'Total Costs',
            'Cost to Charge Ratio',
            'Cost Charge Difference'
        ]
    ]
)

,Total Charges,Total Costs,Cost to Charge Ratio,Cost Charge Difference
351780,74936.75,3970456.20,52.984099,3895519.45
1506586,120261.44,3979133.10,33.087356,3858871.66
324518,221092.03,2216409.24,10.024826,1995317.21
509483,225230.22,2220060.86,9.856852,1994830.64
1330909,231047.65,2222083.22,9.617424,1991035.57
1700813,232804.56,2221993.74,9.544460,1989189.18
2289007,252426.31,2378728.48,9.423457,2126302.17
3460902,253333.06,2378851.00,9.390211,2125517.94
3258273,253499.65,2379050.72,9.384828,2125551.07
1152037,239556.49,2223577.35,9.282058,1984020.86


In [16]:
df[[
    'Total Charges',
    'Total Costs',
    'Cost to Charge Ratio',
    'Cost Charge Difference'
        ]].isna().sum().sum()

1

In [17]:
df.isna().sum().sum()

1

In [18]:
df.dropna(inplace=True)

In [19]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 4208698 entries, 0 to 4208698
Data columns (total 35 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   Hospital Service Area                object 
 1   Hospital County                      object 
 2   Operating Certificate Number         float64
 3   Permanent Facility Id                float64
 4   Facility Name                        object 
 5   Age Group                            object 
 6   Zip Code - 3 digits                  object 
 7   Gender                               object 
 8   Race                                 object 
 9   Ethnicity                            object 
 10  Length of Stay                       int32  
 11  Type of Admission                    object 
 12  Patient Disposition                  object 
 13  Discharge Year                       int64  
 14  CCSR Diagnosis Code                  object 
 15  CCSR Diagnosis Description           

In [20]:
categorical_cols = [
    'Hospital Service Area', 'Hospital County', 'Age Group', 'Zip Code - 3 digits',
    'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'Patient Disposition',
    'APR Severity of Illness Description', 'APR Risk of Mortality',
    'APR Medical Surgical Description', 'Payment Typology 1'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

df['Discharge Year'] = df['Discharge Year'].astype('int16')
df['APR DRG Code'] = df['APR DRG Code'].astype('int16')
df['APR MDC Code'] = df['APR MDC Code'].astype('int8')
df['APR Severity of Illness Code'] = df['APR Severity of Illness Code'].astype('int8')
df['Is_Neonate'] = df['Is_Neonate'].astype('int8')

df['Operating Certificate Number'] = df['Operating Certificate Number'].astype('int32')
df['Permanent Facility Id'] = df['Permanent Facility Id'].astype('int32')

df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 4208698 entries, 0 to 4208698
Data columns (total 35 columns):
 #   Column                               Dtype   
---  ------                               -----   
 0   Hospital Service Area                category
 1   Hospital County                      category
 2   Operating Certificate Number         int32   
 3   Permanent Facility Id                int32   
 4   Facility Name                        object  
 5   Age Group                            category
 6   Zip Code - 3 digits                  category
 7   Gender                               category
 8   Race                                 category
 9   Ethnicity                            category
 10  Length of Stay                       int32   
 11  Type of Admission                    category
 12  Patient Disposition                  category
 13  Discharge Year                       int16   
 14  CCSR Diagnosis Code                  object  
 15  CCSR Diagnosis Descr

In [21]:
df.dtypes.value_counts()

object      8
float64     4
int8        4
int32       3
category    2
int16       2
category    1
Float64     1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
Name: count, dtype: int64

In [22]:
object_cols = df.select_dtypes(include=['object']).columns

print("Object columns:", len(object_cols))

for col in object_cols:
    print(f"{col}: {df[col].nunique(dropna=False)} unique values")

Object columns: 8
Facility Name: 215 unique values
CCSR Diagnosis Code: 481 unique values
CCSR Diagnosis Description: 481 unique values
CCSR Procedure Code: 321 unique values
CCSR Procedure Description: 321 unique values
APR DRG Description: 332 unique values
APR MDC Description: 48 unique values
Emergency Department Indicator: 2 unique values


In [23]:
print(
    "Memory usage (MB):",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2)
)

Memory usage (MB): 2682.4


In [24]:
categorical_cols = [
    'Facility Name',
    'CCSR Diagnosis Code',
    'CCSR Diagnosis Description',
    'CCSR Procedure Code',
    'CCSR Procedure Description',
    'APR DRG Description',
    'APR MDC Description',
    'Emergency Department Indicator'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

print(df[categorical_cols].dtypes)

Facility Name                     category
CCSR Diagnosis Code               category
CCSR Diagnosis Description        category
CCSR Procedure Code               category
CCSR Procedure Description        category
APR DRG Description               category
APR MDC Description               category
Emergency Department Indicator    category
dtype: object


In [ ]:
ccsr_map = {
    'BLD': 'Blood Diseases',
    'CIR': 'Circulatory System Diseases',
    'DEN': 'Dental Diseases',
    'DIG': 'Digestive System Diseases',
    'EAR': 'Ear Diseases',
    'END': 'Endocrine, Nutritional, Metabolic Diseases',
    'EXT': 'External Causes of Morbidity',
    'EYE': 'Eye Diseases',
    'FAC': 'Factors Influencing Health Status',
    'GEN': 'Genitourinary System Diseases',
    'INF': 'Infectious and Parasitic Diseases',
    'INJ': 'Injury and Poisoning',
    'MAL': 'Congenital Malformations',
    'MBD': 'Mental, Behavioral, Neurodevelopmental Disorders',
    'MUS': 'Musculoskeletal System Diseases',
    'NEO': 'Neoplasms',
    'NVS': 'Nervous System Diseases',
    'PNL': 'Perinatal Period Conditions',
    'PRG': 'Pregnancy, Childbirth, Puerperium',
    'RSP': 'Respiratory System Diseases',
    'SKN': 'Skin Diseases',
    'SYM': 'Symptoms, Signs, Abnormal Findings',
    'XXX': 'Unacceptable Principal Diagnosis',
}

col = 'CCSR Diagnosis Code'

cats = df[col].cat.categories
prefix = cats.astype(str).str[:3].str.upper()

unmatched = sorted(set(cats[~prefix.isin(ccsr_map.keys())]))
print("Unmatched codes:", unmatched[:20])

new_names = [ccsr_map.get(p, c) for p, c in zip(prefix, cats.astype(str))]

codes = df[col].cat.codes.to_numpy()
new_cats = pd.Index(new_names)
unique_cats = new_cats.unique()
remap = unique_cats.get_indexer(new_cats) 
new_codes = np.where(codes >= 0, remap[codes], -1)

df[col] = pd.Categorical.from_codes(new_codes, categories=unique_cats)

print(df[col].value_counts(dropna=False))

Unmatched codes: []
CCSR Diagnosis Code
Circulatory System Diseases                         576303
Pregnancy, Childbirth, Puerperium                   431218
Perinatal Period Conditions                         397551
Infectious and Parasitic Diseases                   383641
Digestive System Diseases                           370005
Injury and Poisoning                                321958
Mental, Behavioral, Neurodevelopmental Disorders    294707
Respiratory System Diseases                         256846
Endocrine, Nutritional, Metabolic Diseases          196195
Neoplasms                                           191392
Genitourinary System Diseases                       187727
Musculoskeletal System Diseases                     185189
Nervous System Diseases                             135589
Symptoms, Signs, Abnormal Findings                   90493
Skin Diseases                                        65049
Blood Diseases                                       52923
Factors Influenc

In [26]:
df['CCSR Diagnosis Code'].value_counts()

CCSR Diagnosis Code
Circulatory System Diseases                         576303
Pregnancy, Childbirth, Puerperium                   431218
Perinatal Period Conditions                         397551
Infectious and Parasitic Diseases                   383641
Digestive System Diseases                           370005
Injury and Poisoning                                321958
Mental, Behavioral, Neurodevelopmental Disorders    294707
Respiratory System Diseases                         256846
Endocrine, Nutritional, Metabolic Diseases          196195
Neoplasms                                           191392
Genitourinary System Diseases                       187727
Musculoskeletal System Diseases                     185189
Nervous System Diseases                             135589
Symptoms, Signs, Abnormal Findings                   90493
Skin Diseases                                        65049
Blood Diseases                                       52923
Factors Influencing Health Status   

In [27]:
print(df['Type of Admission'].value_counts(normalize=True))

print(df.groupby('Type of Admission', observed=True)['Length of Stay']
        .agg(['count', 'median', 'mean']))

pd.crosstab(df['Type of Admission'], df['Emergency Department Indicator'],
            normalize='index')

Type of Admission
Emergency        0.664348
Elective         0.168848
Newborn          0.091368
Urgent           0.070777
Trauma           0.003648
Not Available    0.001012
Name: proportion, dtype: float64
                     count  median      mean
Type of Admission                           
Elective            710630     2.0  4.660634
Emergency          2796038     4.0  6.313358
Newborn             384540     2.0  3.535216
Not Available         4261     5.0  8.323164
Trauma               15352     4.0  7.234953
Urgent              297877     3.0  6.778748


Emergency Department Indicator,N,Y
Type of Admission,,
Elective,0.978252,0.021748
Emergency,0.072742,0.927258
Newborn,0.999066,0.000934
Not Available,0.886646,0.113354
Trauma,0.069177,0.930823
Urgent,0.857693,0.142307


In [ ]:
import numpy as np
import pandas as pd

col = 'CCSR Procedure Code'
NO_PROC = {'No Code', 'NO CODE', 'No Procedure', ''}
RARE_THRESHOLD = 50

def remap_categories(series, func):
    """بيطبّق func على الـ categories بس (مئات القيم) وبيوزعها على الصفوف."""
    cats = series.cat.categories
    new_vals = pd.Index([func(c) for c in cats])
    uniq = new_vals.unique()
    remap = uniq.get_indexer(new_vals)
    codes = series.cat.codes.to_numpy()
    new_codes = np.where(codes >= 0, remap[codes], -1)
    return pd.Categorical.from_codes(new_codes, categories=uniq)

# 1) Has_Procedure  (NaN أو علامة الغياب = مفيش إجراء)
no_proc_cats = [c for c in df[col].cat.categories if str(c).strip() in NO_PROC]
codes = df[col].cat.codes.to_numpy()
is_missing_code = np.isin(codes, [df[col].cat.categories.get_loc(c) for c in no_proc_cats])
df['Has_Procedure'] = (~((codes < 0) | is_missing_code)).astype('int8')

def to_prefix(c):
    c = str(c).strip()
    return 'No Procedure' if c in NO_PROC else c[:3].upper()

df['Procedure Prefix'] = remap_categories(df[col], to_prefix)

counts = df[col].value_counts()
rare = set(counts[counts < RARE_THRESHOLD].index)

def group_rare(c):
    return 'Other' if c in rare else str(c)

df['Procedure Code Grouped'] = remap_categories(df[col], group_rare)

print(df['Has_Procedure'].value_counts(normalize=True))
print(df['Procedure Prefix'].nunique(), 'prefixes |',
      df['Procedure Code Grouped'].nunique(), 'grouped codes')

print(df.groupby('Procedure Prefix', observed=True)['Length of Stay']
        .agg(['count', 'median', 'mean'])
        .sort_values('median'))

Has_Procedure
1    0.718041
0    0.281959
Name: proportion, dtype: float64
32 prefixes | 309 grouped codes
                    count  median       mean
Procedure Prefix                            
OST                    20     1.0   3.500000
PNS                  5345     2.0   4.013096
PGN                384704     2.0   2.736545
ENP                  6713     2.0   3.783554
ENT                 48758     2.0   3.439559
EYP                  2013     2.0   5.607551
FRS                 29823     2.0   3.106529
MRS                 89329     2.0   2.609253
ADM                540473     3.0   5.378393
SKB                 38012     3.0   6.667053
No Procedure      1186679     3.0   4.563937
MAM                126837     3.0   5.312109
EST                 15618     3.0   4.857472
HEP                 64694     4.0   6.305608
PLC                  4634     4.0   5.662710
OTR                 73974     4.0   6.309041
GNR                 29359     4.0   5.929970
NCM                   858     4.0   5.

In [29]:
df

,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,...,Emergency Department Indicator,Total Charges,Total Costs,Is_Neonate,Has_Secondary_Insurance,Cost to Charge Ratio,Cost Charge Difference,Has_Procedure,Procedure Prefix,Procedure Code Grouped
0,New York City,Manhattan,7002021,1454,Metropolitan Hospital Center,18 to 29,100,M,Other Race,Spanish/Hispanic,...,Y,21943.33,19793.21,0,0,0.902015,-2150.12,1,OTR,OTR004
1,New York City,Queens,7003000,1626,Elmhurst Hospital Center,50 to 69,113,M,White,Unknown,...,Y,39527.97,18372.44,0,0,0.464796,-21155.53,0,No Procedure,No Code
2,Hudson Valley,Rockland,4324000,776,Montefiore Nyack,70 or Older,109,M,Other Race,Unknown,...,Y,102926.99,25977.53,0,1,0.252388,-76949.46,0,No Procedure,No Code
3,Hudson Valley,Rockland,4324000,776,Montefiore Nyack,50 to 69,109,M,White,Not Span/Hispanic,...,Y,85344.09,18830.76,0,0,0.220645,-66513.33,0,No Procedure,No Code
4,Hudson Valley,Rockland,4324000,776,Montefiore Nyack,70 or Older,109,F,White,Not Span/Hispanic,...,Y,218623.85,42449.25,0,1,0.194166,-176174.60,1,ESA,ESA003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4208694,New York City,Queens,7003004,3376,Cohen Children's Medical Center,0 to 17,115,M,Other Race,Not Span/Hispanic,...,Y,27074.71,4718.63,0,1,0.174282,-22356.08,0,No Procedure,No Code
4208695,New York City,Queens,7003004,3376,Cohen Children's Medical Center,0 to 17,114,F,Other Race,Not Span/Hispanic,...,N,216053.15,27000.66,1,1,0.124972,-189052.49,0,No Procedure,No Code
4208696,New York City,Queens,7003004,3376,Cohen Children's Medical Center,0 to 17,109,F,White,Not Span/Hispanic,...,Y,24867.76,4291.06,0,1,0.172555,-20576.70,0,No Procedure,No Code
4208697,New York City,Queens,7003004,3376,Cohen Children's Medical Center,0 to 17,117,M,White,Spanish/Hispanic,...,N,46873.00,6210.75,0,1,0.132502,-40662.25,0,No Procedure,No Code


In [30]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 4208698 entries, 0 to 4208698
Data columns (total 38 columns):
 #   Column                               Dtype   
---  ------                               -----   
 0   Hospital Service Area                category
 1   Hospital County                      category
 2   Operating Certificate Number         int32   
 3   Permanent Facility Id                int32   
 4   Facility Name                        category
 5   Age Group                            category
 6   Zip Code - 3 digits                  category
 7   Gender                               category
 8   Race                                 category
 9   Ethnicity                            category
 10  Length of Stay                       int32   
 11  Type of Admission                    category
 12  Patient Disposition                  category
 13  Discharge Year                       int16   
 14  CCSR Diagnosis Code                  category
 15  CCSR Diagnosis Descr

In [31]:
df.to_parquet('cleaned_hospital_data_final.parquet', index=False)

In [32]:
df.to_csv('cleaned_hospital_data.csv', index=False)
print("CSV file created successfully!")

CSV file created successfully!
